Step 1: Initializing the Local LLM

In [1]:
import pandas as pd
import json

from langchain_ollama import OllamaLLM
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.prompts import PromptTemplate

# 1. Initialize Local LLM
llm = OllamaLLM(
    model="llama3",
    temperature=0   # results to be consistent and factual
)

Step 2: The Internal Tool (Data Analysis)

In [3]:
def analyze_sales_csv(file_path: str) -> dict:
    df = pd.read_csv(file_path)

    # Calculate revenue per order
    df["order_revenue"] = (
        df["quantity"]
        * df["unit_price"]
        * (1 - df["discount_percent"] / 100)
    )

    # Return structured metrics
    return {
        "total_revenue": round(df["order_revenue"].sum(), 2),
        "average_order_value": round(df["order_revenue"].mean(), 2),
        "orders": len(df),
        "revenue_by_category": (
            df.groupby("product_category")["order_revenue"]
            .sum()
            .round(2)
            .to_dict()
        ),
        "revenue_by_region": (
            df.groupby("region")["order_revenue"]
            .sum()
            .round(2)
            .to_dict()
        ),
        "top_payment_method": (
            df.groupby("payment_method")["order_revenue"]
            .sum()
            .idxmax()
        ),
    }

Step 3: The External Tool (Web Search)

In [5]:
search = DuckDuckGoSearchRun()

def fetch_benchmarks():
    return search.run(
        "average ecommerce order value benchmark"
    )

Step 4: The Orchestration Stage

In [7]:
# Collect all the context before starting up the LLM
def run_analysis():
    # Update this path to where your actual sales.csv lives
    metrics = analyze_sales_csv("sales.csv")
    benchmarks = fetch_benchmarks()

    print("\nDEBUG: METRICS USED\n")
    print(json.dumps(metrics, indent=2))

    return metrics, benchmarks

Step 5: The Prompt Engineering and Execution

In [10]:
writer_prompt = PromptTemplate.from_template("""
You are a business analyst.

RULES:
- Use ONLY the numbers provided
- Do NOT invent values
- Do NOT assume currency
- Do NOT recalculate

METRICS (JSON):
{metrics}

INDUSTRY CONTEXT:
{benchmarks}

Write 3 insights and 3 recommendations.
""")

# MAIN EXECUTION
if __name__ == "__main__":
    # Gather data from our tools
    metrics, benchmarks = run_analysis()

    # Pass the data to the LLM
    response = llm.invoke(
        writer_prompt.format(
            metrics=json.dumps(metrics, indent=2),
            benchmarks=benchmarks
        )
    )

    print("\nFINAL OUTPUT:\n")
    print(response)


DEBUG: METRICS USED

{
  "total_revenue": 322400.0,
  "average_order_value": 32240.0,
  "orders": 10,
  "revenue_by_category": {
    "Electronics": 205700.0,
    "Fashion": 26700.0,
    "Home Appliances": 90000.0
  },
  "revenue_by_region": {
    "East": 52200.0,
    "North": 96700.0,
    "South": 110500.0,
    "West": 63000.0
  },
  "top_payment_method": "Credit Card"
}


ResponseError: model requires more system memory (4.6 GiB) than is available (2.4 GiB) (status code: 500)